# Notebook 01b — Embedding Extraction with Picek's Pretrained ViT-B/16-224

**Goal**: Extract frozen embeddings using the ViT-B/16-224 checkpoint from Picek et al. (2022), which was fine-tuned on DF20-Mini. This replaces our previous CLIP-B/32 embedding.

## Why this works for our study

Picek's ViT was full fine-tuned on DF20-Mini's training set. Using it as a frozen feature extractor gives us:
- Embeddings already specialized for the fungi taxonomy
- Near-SOTA species representation (~0.72 Top-1) without any training on our side
- Direct alignment with the DF20 benchmark

## Important caveat (for honesty)

Since the checkpoint was trained on the same DF20-Mini training split we use, the embedding quality represents an **upper bound**, not a pure transfer learning setup. We document this in the report as a limitation.

## Pipeline
1. Download ViT-B/16-224 checkpoint from Picek's server
2. Load with `timm`, strip classification head
3. Extract 768-dim embeddings from 36k images
4. Save to `artifacts/df20m_embeddings_picekvit.npy`
5. Quick LR sanity check


In [1]:
# ==================== IMPORTS ====================
import os
import urllib.request
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import timm

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"timm version: {timm.__version__}")

/Users/lucilu/miniconda3/envs/mushroom-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
timm version: 1.0.26


In [2]:
# ==================== CONFIG ====================
MODEL_NAME = 'vit_base_patch16_224'
CHECKPOINT_FILENAME = 'DF20M-ViT_base_patch16_224_best_accuracy.pth'
IMAGE_SIZE = 224
EMBED_DIM = 768
NUM_CLASSES_TRAINED = 182   # Picek's DF20-Mini has 182 species per paper (close to our 179)

DATA_DIR      = Path('data/DF20M')
METADATA_DIR  = Path('data/DF20M-metadata')
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

CHECKPOINT_URL = f'http://ptak.felk.cvut.cz/plants/DanishFungiDataset/checkpoints/DF20M/{CHECKPOINT_FILENAME}'
CHECKPOINT_PATH = ARTIFACTS_DIR / CHECKPOINT_FILENAME
EMBED_OUT  = ARTIFACTS_DIR / 'df20m_embeddings_picekvit.npy'
META_OUT   = ARTIFACTS_DIR / 'df20m_metadata_picekvit.csv'

BATCH_SIZE = 16     # conservative for M4; can raise to 32+ on 4060 Ti
NUM_WORKERS = 0     # macOS Jupyter: keep 0

# ImageNet normalization (timm standard)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print(f"Model:       {MODEL_NAME}")
print(f"Input size:  {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Embed dim:   {EMBED_DIM}")
print(f"Output file: {EMBED_OUT}")

Model:       vit_base_patch16_224
Input size:  224x224
Embed dim:   768
Output file: artifacts/df20m_embeddings_picekvit.npy


## Step 1 — Download Checkpoint

In [3]:
if not CHECKPOINT_PATH.exists():
    print(f"Downloading {CHECKPOINT_FILENAME} ...")
    def progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        pct = min(100, 100 * downloaded / total_size)
        if block_num % 100 == 0:
            print(f"\r  {downloaded/1e6:6.1f} / {total_size/1e6:.1f} MB ({pct:5.1f}%)", end='', flush=True)
    urllib.request.urlretrieve(CHECKPOINT_URL, CHECKPOINT_PATH, reporthook=progress)
    print(f"\n Saved to {CHECKPOINT_PATH}")
else:
    print(f"Already exists: {CHECKPOINT_PATH}")
    print(f"  Size: {CHECKPOINT_PATH.stat().st_size/1e6:.1f} MB")

   343.2 / 343.8 MB ( 99.8%)
 Saved to artifacts/DF20M-ViT_base_patch16_224_best_accuracy.pth


## Step 2 — Load Model with Pretrained Weights

In [4]:
# Create ViT with matching head size, then load weights
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES_TRAINED)

state_dict = torch.load(CHECKPOINT_PATH, map_location='cpu')

# Handle wrapped state_dicts
if isinstance(state_dict, dict):
    for key in ('state_dict', 'model_state_dict', 'model'):
        if key in state_dict and isinstance(state_dict[key], dict):
            state_dict = state_dict[key]
            print(f"Unwrapped state_dict from key '{key}'")
            break

# Strip DataParallel prefix if present
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

# Load; allow mismatched head (our 179 vs their 182)
try:
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    print("Loaded strictly")
except RuntimeError as e:
    print(f"Strict load failed: {str(e)[:150]}")
    print("Retrying with strict=False ...")
    load_result = model.load_state_dict(state_dict, strict=False)
    print(f"Missing keys:    {len(load_result.missing_keys)}")
    print(f"Unexpected keys: {len(load_result.unexpected_keys)}")
    if load_result.missing_keys:
        print(f"  Missing sample:    {load_result.missing_keys[:3]}")
    if load_result.unexpected_keys:
        print(f"  Unexpected sample: {load_result.unexpected_keys[:3]}")

# Strip classification head - we want pre-logits features
model.reset_classifier(0)
model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)

# Sanity forward pass
with torch.no_grad():
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    feat = model(dummy)
    print(f"\nOutput feature shape: {feat.shape}")
    assert feat.shape == (1, EMBED_DIM), f"Expected (1, {EMBED_DIM}), got {feat.shape}"
print("Model ready for embedding extraction")

Loaded strictly

Output feature shape: torch.Size([1, 768])
Model ready for embedding extraction


## Step 3 — Load Metadata

In [6]:
# Load official splits
train_meta = pd.read_csv(METADATA_DIR / 'DF20M-train_metadata_PROD.csv')
test_meta  = pd.read_csv(METADATA_DIR / 'DF20M-public_test_metadata_PROD.csv')
train_meta['split'] = 'train'
test_meta['split']  = 'test'

metadata = pd.concat([train_meta, test_meta], ignore_index=True)
print(f"Total rows: {len(metadata):,}")
print(f"Columns: {metadata.columns.tolist()[:15]}")

# Identify image path column
IMAGE_COL = None
for candidate in ['image_path', 'filename', 'file_name', 'image', 'image_filename', 'ImageUniqueID']:
    if candidate in metadata.columns:
        IMAGE_COL = candidate
        break
if IMAGE_COL is None:
    raise ValueError(f"No image column found. Available: {metadata.columns.tolist()}")
print(f"\nImage column: '{IMAGE_COL}'")
print(f"Example value: {metadata[IMAGE_COL].iloc[0]}")

Total rows: 36,393
Columns: ['gbifID', 'eventDate', 'year', 'month', 'day', 'countryCode', 'locality', 'taxonID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus']

Image column: 'image_path'
Example value: 2862684394-136762.JPG


In [ ]:
# ---- Option: reuse the already-cleaned metadata from previous pipeline ----
# This metadata already has species_id, split, and was cleaned of NaNs.
# Same image ordering as the old pipeline.

OLD_META_PATH = ARTIFACTS_DIR / 'df20m_metadata.csv'

if OLD_META_PATH.exists():
    print(f"Reusing clean metadata from: {OLD_META_PATH}")
    metadata = pd.read_csv(OLD_META_PATH)
    metadata['species_id']    = metadata['species_id'].astype(int)
    metadata['embedding_idx'] = metadata['embedding_idx'].astype(int)
    
    # identify image column
    IMAGE_COL = None
    for candidate in ['image_path', 'filename', 'file_name', 'image', 'image_filename', 'ImageUniqueID']:
        if candidate in metadata.columns:
            IMAGE_COL = candidate
            break
    if IMAGE_COL is None:
        raise ValueError(f"No image column found. Available: {metadata.columns.tolist()}")
    
    print(f"Total rows: {len(metadata):,}")
    print(f"Image column: '{IMAGE_COL}'")
    print(f"Example image path: {metadata[IMAGE_COL].iloc[0]}")
    print(f"\nSplits:")
    print(metadata['split'].value_counts())

else:
    # ---- Fallback: load Picek raw CSVs and clean ----
    print("Old metadata not found, loading Picek's raw CSVs ...")
    train_meta = pd.read_csv(METADATA_DIR / 'DF20M-train_metadata_PRODUCTION.csv')
    test_meta  = pd.read_csv(METADATA_DIR / 'DF20M-public_test_metadata_PRODUCTION.csv')
    train_meta['split'] = 'train'
    test_meta['split']  = 'test'
    metadata = pd.concat([train_meta, test_meta], ignore_index=True)
    
    # Drop NaN species
    n_missing = metadata['species'].isna().sum()
    if n_missing > 0:
        print(f"Dropping {n_missing} NaN-species rows")
        metadata = metadata.dropna(subset=['species']).reset_index(drop=True)
    
    # Identify image column
    IMAGE_COL = None
    for candidate in ['image_path', 'filename', 'file_name', 'image', 'image_filename', 'ImageUniqueID']:
        if candidate in metadata.columns:
            IMAGE_COL = candidate
            break
    if IMAGE_COL is None:
        raise ValueError(f"No image column found. Available: {metadata.columns.tolist()}")
    
    # Build species_id
    species_list = sorted(metadata['species'].unique())
    species_to_id = {s: i for i, s in enumerate(species_list)}
    metadata['species_id'] = metadata['species'].map(species_to_id)
    metadata['embedding_idx'] = metadata.index
    
    print(f"Total rows: {len(metadata):,}")
    print(f"Species: {len(species_list)}")
    print(f"Image column: '{IMAGE_COL}'")

# Save for downstream use
metadata.to_csv(META_OUT, index=False)
print(f"\nSaved metadata: {META_OUT}")

TypeError: '<' not supported between instances of 'float' and 'str'

## Step 4 — Dataset + DataLoader

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


class MushroomDataset(Dataset):
    def __init__(self, metadata_df, data_dir, image_col, transform):
        self.df = metadata_df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.image_col = image_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row[self.image_col]
        img_path = Path(img_name)
        if not img_path.is_absolute():
            img_path = self.data_dir / img_name
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
        return self.transform(img), idx


dataset = MushroomDataset(metadata, DATA_DIR, IMAGE_COL, preprocess)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS,
                     pin_memory=torch.cuda.is_available())
print(f"Dataset size: {len(dataset):,}")
print(f"Batches: {len(loader):,}")

# Test load a single image to verify paths work
test_img, test_idx = dataset[0]
print(f"Sample image tensor shape: {test_img.shape}")

## Step 5 — Extract Embeddings

On M4 MPS, this takes ~20-40 min for 36k images at 224px.  
On 4060 Ti, ~5-10 min.


In [ ]:
all_embeddings = np.zeros((len(dataset), EMBED_DIM), dtype=np.float32)

t0 = time.time()
model.eval()
with torch.no_grad():
    for batch_imgs, batch_indices in tqdm(loader, desc='Extracting', total=len(loader)):
        batch_imgs = batch_imgs.to(DEVICE, non_blocking=True)
        feats = model(batch_imgs)
        all_embeddings[batch_indices.numpy()] = feats.cpu().numpy()

elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")
print(f"Shape: {all_embeddings.shape}")
print(f"Memory: {all_embeddings.nbytes/1e6:.1f} MB")
print(f"Mean embedding norm: {np.linalg.norm(all_embeddings, axis=1).mean():.2f}")

In [ ]:
# L2-normalize (important for cosine-based coreset strategy)
norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
all_embeddings_normalized = all_embeddings / (norms + 1e-8)

np.save(EMBED_OUT, all_embeddings_normalized)
metadata.to_csv(META_OUT, index=False)
print(f"Saved embeddings: {EMBED_OUT}")
print(f"  Size: {EMBED_OUT.stat().st_size/1e6:.1f} MB")
print(f"Saved metadata:   {META_OUT}")

## Step 6 — Quick Sanity Check (Logistic Regression)

If the checkpoint loaded correctly, a simple LR on these embeddings should reach **0.60-0.72 species test accuracy** (compared to our previous CLIP+LR result of 0.17).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_mask = metadata['split'].values == 'train'
test_mask  = metadata['split'].values == 'test'

X_train = all_embeddings_normalized[train_mask]
y_train = metadata.loc[train_mask, 'species_id'].to_numpy()
X_test  = all_embeddings_normalized[test_mask]
y_test  = metadata.loc[test_mask, 'species_id'].to_numpy()

print(f"X_train: {X_train.shape}, y_train classes: {len(np.unique(y_train))}")
print(f"X_test:  {X_test.shape},  y_test classes: {len(np.unique(y_test))}")
print(f"\nFitting LR on {len(X_train):,} samples, {X_train.shape[1]}-dim features ...")

t0 = time.time()
lr = LogisticRegression(max_iter=500, C=10.0, n_jobs=-1, solver='lbfgs')
lr.fit(X_train, y_train)
fit_time = time.time() - t0

y_pred_train = lr.predict(X_train)
y_pred_test  = lr.predict(X_test)

print(f"LR fit time: {fit_time:.1f}s")
print(f"\nSPECIES ACCURACY (Picek ViT-B/16-224 + LR):")
print(f"  Train: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"  Test:  {accuracy_score(y_test,  y_pred_test):.3f}")

print(f"\nComparison points:")
print(f"  Frozen CLIP-B/32 + LR (our V1):     0.170")
print(f"  This (Picek ViT + LR):              [see above]")
print(f"  Picek ViT-B/16-224 full FT (their): ~0.720")
print(f"  Picek ViT-L/16-384 SOTA:            ~0.758")
print(f"\nExpected range for this cell: 0.60 - 0.72")
print(f"If < 0.40, there's a checkpoint loading issue. Debug before proceeding.")

## Step 7 — Ready for Notebook 02b

Next step: Open `02b_active_learning_picekvit.ipynb` to run the full AL pipeline on these embeddings.

Key numbers to report back:
1. **LR test accuracy** (from Step 6) — validates embedding quality
2. **Extraction time** — for the report's methodology section

If everything looks good, proceed to AL experiments.
